# 08 · Power BI Dashboard (generated as a Power BI Project)

Builds a complete **Power BI Project (.pbip)** from code, so the dashboard is reproducible and reviewable:

```
powerbi/BatteryInsights/
├── BatteryInsights.pbip              ← open this in Power BI Desktop
├── BatteryInsights.SemanticModel/    ← TMDL: tables, measures, relationships, CSV queries
└── BatteryInsights.Report/           ← PBIR: 1 dashboard page, every visual as JSON, custom theme
```

**One page: EV Battery Analytics, covering state-of-charge estimation, energy consumption prediction and anomaly detection**
- **KPI row:** trips, consumption, heating and regeneration shares, SoC model error, anomaly flags.
- **Energy row:** consumption vs temperature, where the energy goes, winter warm-up, the cold-battery regeneration limit.
- **Battery row:** pack resistance vs temperature, the virtual SoC sensor on an unseen trip, and the anomaly register.

Every report JSON file is validated against Microsoft's published schemas before the notebook finishes.
After opening the project, click **Refresh** once to load the CSVs from notebook 07.

In [1]:
from pathlib import Path
import json
import shutil
import uuid

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "powerbi" / "data"
NAME = "BatteryInsights"
OUT = ROOT / "powerbi" / NAME
DATA_DIR = str(DATA.resolve()) + "\\"
NS = uuid.UUID("3b8e2f4a-7c1d-4e5f-9a0b-1c2d3e4f5a6b")


def tag(*parts):
    # deterministic GUID: regenerating the project gives identical files
    return str(uuid.uuid5(NS, "|".join(map(str, parts))))


def short_id(*parts):
    return uuid.uuid5(NS, "|".join(map(str, parts))).hex[:20]


if OUT.exists():
    shutil.rmtree(OUT)
print("building", OUT, "| data folder", DATA_DIR)

building D:\M.TECH\Companies\volvo\powerbi\BatteryInsights | data folder D:\M.TECH\Companies\volvo\powerbi\data\


## 1 · Semantic model

In [2]:
DESCRIPTIONS = {
    "dim_trip": "One row per trip (BMW i3, Munich, 2019-2020).",
    "fact_trip": "Per-trip energy, distance, climate load and anomaly scores.",
    "fact_sag_event": "Voltage-sag events with apparent and expected pack resistance.",
    "fact_warmup": "Mean heating power and cabin temperature per minute since start (winter trips).",
    "fact_regen_limit": "99th-percentile regeneration current per battery-temperature band.",
    "fact_soc_test": "SoC recorded vs model estimate on unseen test trips (5 s steps).",
    "fact_model": "Model accuracy per task, model and split.",
    "fact_driver": "Importance of each input per model.",
    "fact_anomaly": "Anomaly register with root-cause notes.",
}


def column_types(path):
    df = pd.read_csv(path)
    out = []
    for c, dt in df.dtypes.items():
        out.append((c, "int" if pd.api.types.is_integer_dtype(dt) else "num" if pd.api.types.is_float_dtype(dt) else "text"))
    return out


TABLES = {t: (DESCRIPTIONS[t], column_types(DATA / f"{t}.csv")) for t in DESCRIPTIONS}
HIDDEN = {("fact_regen_limit", "band_low_c")}
SORT_BY = {("fact_regen_limit", "batt_temp_band"): "band_low_c"}
TEXT_KEYS = {"event_id"}  # numeric ids used as categories
COLUMN_FORMAT = {"share": "0.0%"}

RELATIONSHIPS = [
    ("fact_trip", "trip_id", "dim_trip", "trip_id"),
    ("fact_sag_event", "trip_id", "dim_trip", "trip_id"),
    ("fact_soc_test", "trip_id", "dim_trip", "trip_id"),
    ("fact_anomaly", "trip_id", "dim_trip", "trip_id"),
]

PER_100 = "DIVIDE ( {num}, SUM ( fact_trip[distance_km] ) ) * 100"
MEASURES = [
    # (home table, name, DAX, format, folder, description)
    ("dim_trip", "Trips", "COUNTROWS ( dim_trip )", "#,0", "Energy", "Number of trips."),
    ("fact_trip", "Distance km", "SUM ( fact_trip[distance_km] )", "#,0", "Energy", "Distance integrated from speed."),
    ("fact_trip", "Consumption", PER_100.format(num="SUM ( fact_trip[energy_net_kwh] )"), "0.0", "Energy",
     "Net battery energy per 100 km (drawn minus recovered), distance-weighted."),
    ("fact_trip", "Traction", PER_100.format(num="SUM ( fact_trip[energy_net_kwh] ) - SUM ( fact_trip[heating_kwh] ) - SUM ( fact_trip[aircon_kwh] )"),
     "0.0", "Energy", "Consumption without heating and A/C, kWh per 100 km."),
    ("fact_trip", "Heating", PER_100.format(num="SUM ( fact_trip[heating_kwh] )"), "0.0", "Energy",
     "Cabin heating energy per 100 km."),
    ("fact_trip", "Air conditioning", PER_100.format(num="SUM ( fact_trip[aircon_kwh] )"), "0.0", "Energy",
     "A/C energy per 100 km."),
    ("fact_trip", "Heating Share", "DIVIDE ( SUM ( fact_trip[heating_kwh] ), SUM ( fact_trip[energy_out_kwh] ) )",
     "0.0%", "Energy", "Share of battery energy spent on cabin heating."),
    ("fact_trip", "Regen Share", "DIVIDE ( SUM ( fact_trip[energy_regen_kwh] ), SUM ( fact_trip[energy_out_kwh] ) )",
     "0.0%", "Energy", "Share of drawn energy recovered by regenerative braking."),
    ("fact_trip", "Ambient Temp", "AVERAGE ( fact_trip[ambient_mean_c] )", "0.0", "Energy",
     "Mean ambient temperature while driving."),
    ("fact_sag_event", "Sag Events", "COUNTROWS ( fact_sag_event )", "#,0", "Battery", "Voltage-sag events."),
    ("fact_sag_event", "Median Resistance", "MEDIAN ( fact_sag_event[resistance_mohm] )", "0", "Battery",
     "Median apparent pack resistance (milliohm)."),
    ("fact_sag_event", "Resistance", "AVERAGE ( fact_sag_event[resistance_mohm] )", "0", "Battery",
     "Apparent pack resistance (milliohm)."),
    ("fact_sag_event", "Battery Temp", "AVERAGE ( fact_sag_event[batt_temp_c] )", "0.0", "Battery",
     "Battery temperature at the event."),
    ("fact_warmup", "Heater Power", "AVERAGE ( fact_warmup[heating_kw] )", "0.0", "Energy", "Mean heating power (kW)."),
    ("fact_warmup", "Cabin Temp", "AVERAGE ( fact_warmup[cabin_temp_c] )", "0.0", "Energy", "Mean cabin temperature."),
    ("fact_regen_limit", "Regen Limit", "MAX ( fact_regen_limit[regen_current_p99_a] )", "0", "Battery",
     "99th-percentile regeneration current (A)."),
    ("fact_soc_test", "SoC Recorded", "AVERAGE ( fact_soc_test[soc_recorded] )", "0.0", "Models", "SoC logged by the car."),
    ("fact_soc_test", "SoC Estimated", "AVERAGE ( fact_soc_test[soc_estimated] )", "0.0", "Models",
     "SoC estimated by the model from voltage, current and temperature."),
    ("fact_model", "SoC Model Error",
     "CALCULATE ( MAX ( fact_model[mae] ), fact_model[task] = \"A · SoC estimation\", fact_model[split] = \"test\", fact_model[is_selected] = 1 )",
     "0.0", "Models", "Mean absolute SoC error on unseen test trips (percentage points)."),
    ("fact_model", "Energy Model Error",
     "CALCULATE ( MAX ( fact_model[mae] ), fact_model[task] = \"B · segment energy\", fact_model[split] = \"test\", fact_model[is_selected] = 1 )",
     "0.0", "Models", "Mean absolute error of segment energy on unseen test trips (kWh/100 km)."),
    ("fact_driver", "Driver Share", "SUM ( fact_driver[share] )", "0%", "Models", "Share of total importance."),
    ("fact_anomaly", "Anomaly Flags", "COUNTROWS ( fact_anomaly )", "#,0", "Anomalies", "Flags raised by all detectors."),
    ("fact_anomaly", "Flagged Trips", "DISTINCTCOUNT ( fact_anomaly[trip_id] )", "#,0", "Anomalies", "Trips with at least one flag."),
]
MEASURE_HOME = {name: table for table, name, *_ in MEASURES}
assert len(MEASURE_HOME) == len(MEASURES), "duplicate measure name"
print(f"{len(TABLES)} tables | {sum(len(c) for _, c in TABLES.values())} columns | {len(MEASURES)} measures | "
      f"{len(RELATIONSHIPS)} relationships")

9 tables | 67 columns | 23 measures | 4 relationships


In [3]:
M_TYPE = {"text": "type text", "int": "Int64.Type", "num": "type number"}
TMDL_TYPE = {"text": "string", "int": "int64", "num": "double"}


def q(name):
    return name if name.replace("_", "").isalnum() else "'" + name.replace("'", "''") + "'"


def m_query(table, columns):
    types = ", ".join(f'{{"{c}", {M_TYPE[t]}}}' for c, t in columns)
    return ["let",
            f'    Source = Csv.Document(File.Contents(DataFolder & "{table}.csv"), [Delimiter = ",", Encoding = 65001, QuoteStyle = QuoteStyle.Csv]),',
            "    Headers = Table.PromoteHeaders(Source, [PromoteAllScalars = true]),",
            f'    Typed = Table.TransformColumnTypes(Headers, {{{types}}}, "en-US")',
            "in", "    Typed"]


def table_tmdl(table, description, columns):
    L = [f"/// {description}", f"table {q(table)}", f"\tlineageTag: {tag('table', table)}", ""]
    for home, name, dax, fmt, folder, desc in MEASURES:
        if home != table:
            continue
        L += [f"\t/// {desc}", f"\tmeasure {q(name)} = {dax}"]
        if fmt:
            L.append(f"\t\tformatString: {fmt}")
        L += [f"\t\tdisplayFolder: {folder}", f"\t\tlineageTag: {tag('measure', table, name)}", ""]
    for col, typ in columns:
        L += [f"\tcolumn {q(col)}", f"\t\tdataType: {TMDL_TYPE[typ]}"]
        fmt = COLUMN_FORMAT.get(col, "0" if typ == "int" else None)
        if fmt and typ in ("int", "num"):
            L.append(f"\t\tformatString: {fmt}")
        if (table, col) in HIDDEN:
            L.append("\t\tisHidden")
        L += [f"\t\tlineageTag: {tag('column', table, col)}", "\t\tsummarizeBy: none", f"\t\tsourceColumn: {col}"]
        if (table, col) in SORT_BY:
            L.append(f"\t\tsortByColumn: {SORT_BY[(table, col)]}")
        L += ["", "\t\tannotation SummarizationSetBy = User", ""]
    L += [f"\tpartition {q(table)} = m", "\t\tmode: import", "\t\tsource ="]
    L += [f"\t\t\t\t{line}" for line in m_query(table, columns)]
    L += ["", "\tannotation PBI_ResultType = Table", ""]
    return "\n".join(L)


def write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def write_json(path, obj):
    write(path, json.dumps(obj, indent=2, ensure_ascii=False) + "\n")


SM = OUT / f"{NAME}.SemanticModel"
DEF = SM / "definition"
write(DEF / "database.tmdl", "database\n\tcompatibilityLevel: 1567\n\n")
write(DEF / "model.tmdl", "\n".join([
    "model Model", "\tculture: en-US", "\tdefaultPowerBIDataSourceVersion: powerBI_V3", "\tdiscourageImplicitMeasures",
    "\tsourceQueryCulture: en-US", "\tdataAccessOptions", "\t\tlegacyRedirects", "\t\treturnErrorValuesAsNull", "",
    f"annotation PBI_QueryOrder = {json.dumps(list(TABLES) + ['DataFolder'])}", "",
    "annotation __PBI_TimeIntelligenceEnabled = 0", "", 'annotation PBI_ProTooling = ["DevMode"]', "",
    *[f"ref table {q(t)}" for t in TABLES], ""]))
write(DEF / "expressions.tmdl", "\n".join([
    f'expression DataFolder = "{DATA_DIR}" meta [IsParameterQuery=true, Type="Text", IsParameterQueryRequired=true]',
    f"\tlineageTag: {tag('expression', 'DataFolder')}", "", "\tannotation PBI_ResultType = Text", ""]))
rel_lines = []
for f_table, f_col, t_table, t_col in RELATIONSHIPS:
    rel_lines += [f"relationship {tag('rel', f_table, f_col, t_table)}",
                  f"\tfromColumn: {f_table}.{f_col}", f"\ttoColumn: {t_table}.{t_col}", ""]
write(DEF / "relationships.tmdl", "\n".join(rel_lines))
for table, (description, columns) in TABLES.items():
    write(DEF / "tables" / f"{table}.tmdl", table_tmdl(table, description, columns))
write_json(SM / "definition.pbism", {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/semanticModel/definitionProperties/1.0.0/schema.json",
    "version": "4.2", "settings": {}})
write_json(SM / ".platform", {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
    "metadata": {"type": "SemanticModel", "displayName": NAME},
    "config": {"version": "2.0", "logicalId": tag("logical", "semanticModel")}})
print((DEF / "tables" / "fact_trip.tmdl").read_text(encoding="utf-8")[:900])

/// Per-trip energy, distance, climate load and anomaly scores.
table fact_trip
	lineageTag: 0ba37a31-3a85-5126-ad27-bc72ec3bb6d4

	/// Distance integrated from speed.
	measure 'Distance km' = SUM ( fact_trip[distance_km] )
		formatString: #,0
		displayFolder: Energy
		lineageTag: 46646363-cd08-5126-bbb7-096591ab0bf5

	/// Net battery energy per 100 km (drawn minus recovered), distance-weighted.
	measure Consumption = DIVIDE ( SUM ( fact_trip[energy_net_kwh] ), SUM ( fact_trip[distance_km] ) ) * 100
		formatString: 0.0
		displayFolder: Energy
		lineageTag: 3037574a-3d5e-5b97-82b5-0d5238a05150

	/// Consumption without heating and A/C, kWh per 100 km.
	measure Traction = DIVIDE ( SUM ( fact_trip[energy_net_kwh] ) - SUM ( fact_trip[heating_kwh] ) - SUM ( fact_trip[aircon_kwh] ), SUM ( fact_trip[distance_km] ) ) * 100
		formatString: 0.0
		displayFolder: Energy
		lineageTag: 5f03feb0-afa4-5


## 2 · Design system

| Token | Colour | Used for |
|---|---|---|
| Navy | `#0E1E33` | header band |
| Ink / secondary ink | `#14213D` / `#5B6576` | titles / labels |
| Canvas / card | `#F2F4F8` / `#FFFFFF` | page background / visual cards |
| Winter | `#2D7FF9` | winter trips everywhere |
| Summer | `#F29C38` | summer trips everywhere |
| Heating | `#E4572E` | heating energy, heater power |
| Teal | `#13A89E` | models, regeneration |
| Flag | `#D7263D` | flagged anomalies |

Canvas 1280 × 720 with a 20 px margin and a 12 px gutter. Each page follows the same grid:
header band → KPI row → two rows of two visuals.

In [4]:
SCHEMA = "https://developer.microsoft.com/json-schemas/fabric/item/report/definition"
VC_SCHEMA = f"{SCHEMA}/visualContainer/2.1.0/schema.json"
NAVY, INK, INK2, MUTED = "#0E1E33", "#14213D", "#5B6576", "#8C96A8"
CANVAS, CARD, HAIRLINE, GRID = "#F2F4F8", "#FFFFFF", "#E1E6EE", "#EEF1F5"
WINTER, SUMMER, HEAT, TEAL, FLAG, NEUTRAL = "#2D7FF9", "#F29C38", "#E4572E", "#13A89E", "#D7263D", "#B8C1CE"
W, H, MARGIN, GAP = 1280, 720, 20, 12

THEME = {
    "name": "BatteryInsights",
    "dataColors": [WINTER, SUMMER, TEAL, HEAT, "#7A5CFA", NEUTRAL, NAVY, FLAG],
    "background": CARD, "foreground": INK, "tableAccent": WINTER,
    "good": TEAL, "neutral": SUMMER, "bad": FLAG,
    "textClasses": {
        "callout": {"fontSize": 24, "fontFace": "Segoe UI Semibold", "color": INK},
        "title": {"fontSize": 11, "fontFace": "Segoe UI Semibold", "color": INK},
        "header": {"fontSize": 11, "fontFace": "Segoe UI Semibold", "color": INK},
        "label": {"fontSize": 9, "fontFace": "Segoe UI", "color": INK2},
    },
    "visualStyles": {"*": {"*": {
        "valueAxis": [{"gridlineColor": {"solid": {"color": GRID}}, "labelColor": {"solid": {"color": INK2}},
                       "titleColor": {"solid": {"color": INK2}}, "fontSize": 9}],
        "categoryAxis": [{"labelColor": {"solid": {"color": INK2}}, "titleColor": {"solid": {"color": INK2}}, "fontSize": 9}],
        "legend": [{"labelColor": {"solid": {"color": INK2}}, "fontSize": 9}],
    }}},
}


def lit(value):
    return {"expr": {"Literal": {"Value": value}}}


def text_lit(s):
    return lit("'" + s.replace("'", "''") + "'")


def color(hex_):
    return {"solid": {"color": lit(f"'{hex_}'")}}


def C(ref, label=None):
    table, column = ref.split(".", 1)
    return ({"Column": {"Expression": {"SourceRef": {"Entity": table}}, "Property": column}}, label)


def M(name, label=None):
    return ({"Measure": {"Expression": {"SourceRef": {"Entity": MEASURE_HOME[name]}}, "Property": name}}, label)


def projection(item, active):
    field, label = item
    kind = "Column" if "Column" in field else "Measure"
    entity, prop = field[kind]["Expression"]["SourceRef"]["Entity"], field[kind]["Property"]
    p = {"field": field, "queryRef": f"{entity}.{prop}", "nativeQueryRef": label or prop}
    if label:
        p["displayName"] = label
    if active:
        p["active"] = True
    return p


def value_color(column_ref, value, hex_):
    table, column = column_ref.split(".", 1)
    return {"properties": {"fill": color(hex_)}, "selector": {"data": [{"scopeId": {"Comparison": {
        "ComparisonKind": 0,
        "Left": {"Column": {"Expression": {"SourceRef": {"Entity": table}}, "Property": column}},
        "Right": {"Literal": {"Value": f"'{value}'"}}}}}]}}


def measure_color(name, hex_):
    return {"properties": {"fill": color(hex_)}, "selector": {"metadata": f"{MEASURE_HOME[name]}.{name}"}}


SEASON_COLORS = [value_color("dim_trip.season", "Summer", SUMMER), value_color("dim_trip.season", "Winter", WINTER)]


def in_filter(ref, values, negate=False):
    table, column = ref.split(".", 1)
    cond = {"In": {"Expressions": [{"Column": {"Expression": {"SourceRef": {"Source": "f"}}, "Property": column}}],
                   "Values": [[{"Literal": {"Value": f"'{v}'" if isinstance(v, str) else f"{v}L"}}] for v in values]}}
    if negate:
        cond = {"Not": {"Expression": cond}}
    return {"name": short_id("filter", ref, *values), "field": C(ref)[0], "type": "Categorical",
            "filter": {"Version": 2, "From": [{"Name": "f", "Entity": table, "Type": 0}], "Where": [{"Condition": cond}]}}


def card_frame():
    return {"background": [{"properties": {"show": lit("true"), "color": color(CARD), "transparency": lit("0D")}}],
            "border": [{"properties": {"show": lit("true"), "color": color(HAIRLINE), "radius": lit("10D")}}],
            "dropShadow": [{"properties": {"show": lit("false")}}],
            "padding": [{"properties": {"top": lit("10D"), "bottom": lit("8D"), "left": lit("12D"), "right": lit("12D")}}]}

In [5]:
class Page:
    def __init__(self, key, display_name):
        self.key, self.display_name, self.visuals, self.z = key, display_name, [], 0

    def add(self, vtype, x, y, w, h, roles=None, title=None, subtitle=None, objects=None, sort=None, filters=None,
            frame=True, vco=None):
        self.z += 1
        visual = {"visualType": vtype}
        if roles:
            visual["query"] = {"queryState": {
                role: {"projections": [projection(i, active=role in ("Category", "Rows")) for i in items]}
                for role, items in roles.items()}}
            if sort:
                (field, _), direction = sort
                visual["query"]["sortDefinition"] = {"sort": [{"field": field, "direction": direction}], "isDefaultSort": True}
        if objects:
            visual["objects"] = objects
        container_objects = card_frame() if frame else {}
        container_objects["title"] = [{"properties": {"show": lit("true" if title else "false"), **(
            {"text": text_lit(title), "fontColor": color(INK), "fontSize": lit("11D"), "bold": lit("true"),
             "fontFamily": text_lit("Segoe UI Semibold")} if title else {})}}]
        if subtitle:
            container_objects["subTitle"] = [{"properties": {"show": lit("true"), "text": text_lit(subtitle),
                                                             "fontColor": color(MUTED), "fontSize": lit("9D")}}]
        if vco:
            container_objects.update(vco)
        visual["visualContainerObjects"] = container_objects
        visual["drillFilterOtherVisuals"] = True
        container = {"$schema": VC_SCHEMA, "name": f"{self.key}_{self.z:02d}_{vtype}"[:50],
                     "position": {"x": x, "y": y, "z": self.z * 100, "width": w, "height": h, "tabOrder": self.z * 100},
                     "visual": visual}
        if filters:
            container["filterConfig"] = {"filters": filters}
        self.visuals.append(container)

    def text(self, x, y, w, h, paragraphs, background=None):
        paras = [{"textRuns": [{"value": t, "textStyle": {"fontFamily": "Segoe UI Semibold" if bold else "Segoe UI",
                                                          "fontSize": f"{pt}pt", "color": hex_}}
                               for t, pt, bold, hex_ in runs]} for runs in paragraphs]
        vco = None
        if background:
            vco = {"background": [{"properties": {"show": lit("true"), "color": color(background), "transparency": lit("0D")}}],
                   "padding": [{"properties": {"top": lit("8D"), "left": lit("20D"), "right": lit("20D"), "bottom": lit("4D")}}]}
        self.add("textbox", x, y, w, h, objects={"general": [{"properties": {"paragraphs": paras}}]}, frame=False, vco=vco)

    def header(self, title, subtitle, page_label):
        self.text(0, 0, W, 60, [[(title, 17, True, "#FFFFFF"), ("   " + page_label, 11, False, "#9FB3D1")],
                                [(subtitle, 9, False, "#C8D3E6")]], background=NAVY)
        self.text(MARGIN, H - 30, W - 2 * MARGIN, 26,
                  [[("Data: Battery and Heating Data in Real Driving Cycles (BMW i3, TU Munich, IEEE DataPort, CC BY 4.0) · "
                     "70 trips · 1.09 M samples at 10 Hz · built with Python, DuckDB and Power BI", 8, False, MUTED)]])

    def season_slicer(self, x, y, w, h):
        self.add("slicer", x, y, w, h, roles={"Values": [C("dim_trip.season", "Season")]}, title="Season",
                 objects={"data": [{"properties": {"mode": text_lit("Dropdown")}}],
                          "header": [{"properties": {"show": lit("false")}}],
                          "items": [{"properties": {"fontColor": color(INK), "textSize": lit("10D")}}],
                          "selection": [{"properties": {"selectAllCheckboxEnabled": lit("true"),
                                                        "singleSelect": lit("false")}}]})

    def cards(self, x0, y, h, measures):
        w = (W - MARGIN - x0 - GAP * (len(measures) - 1)) / len(measures)
        for i, (name, label, accent) in enumerate(measures):
            self.add("card", x0 + i * (w + GAP), y, w, h, roles={"Values": [M(name, label)]},
                     objects={"labels": [{"properties": {"fontSize": lit("19D"), "color": color(accent),
                                                         "fontFamily": text_lit("Segoe UI Semibold"),
                                                         "labelDisplayUnits": lit("1D")}}],
                              "categoryLabels": [{"properties": {"show": lit("true"), "color": color(INK2),
                                                                 "fontSize": lit("9D")}}]})

    def json(self):
        return {"$schema": f"{SCHEMA}/page/2.0.0/schema.json", "name": self.key, "displayName": self.display_name,
                "displayOption": "FitToPage", "width": W, "height": H,
                "objects": {"background": [{"properties": {"color": color(CANVAS), "transparency": lit("0D")}}]}}


NO_AXIS_TITLES = {"categoryAxis": [{"properties": {"showAxisTitle": lit("false")}}],
                  "valueAxis": [{"properties": {"showAxisTitle": lit("false")}}]}

## 3 · The dashboard page

In [6]:
models = pd.read_csv(DATA / "fact_model.csv")
chosen = models[models.split.eq("test") & models.is_selected.eq(1)].set_index("task")  # selected model per task
soc_test = pd.read_csv(DATA / "fact_soc_test.csv")
err = soc_test.assign(e=(soc_test.soc_estimated - soc_test.soc_recorded).abs()).groupby("trip_id").e.mean().sort_values()
show_trip = err.index[len(err) // 2]  # the median-error test trip: neither the best nor the worst case
print(f"SoC example trip: {show_trip} (mean error {err[show_trip]:.1f} points, median of {len(err)} test trips)")

TOP = pd.read_csv(DATA / "fact_driver.csv").query("task == 'B · segment energy'").sort_values("share", ascending=False).iloc[0]
n_flags = len(pd.read_csv(DATA / "fact_anomaly.csv"))

HEADER_H = 60
ROW1_Y, ROW1_H = HEADER_H + 10, 80
ROW2_Y = ROW1_Y + ROW1_H + GAP
FOOTER_H = 26
ROW_H = (H - FOOTER_H - 6 - ROW2_Y - GAP) // 2
ROW3_Y = ROW2_Y + ROW_H + GAP
SLICER_W = 150
CONTENT_W = W - 2 * MARGIN
Q_W = (CONTENT_W - 3 * GAP) / 4          # four tiles in the energy row
B_W = 352                                  # resistance and SoC tiles in the bottom row
TABLE_X = MARGIN + 2 * (B_W + GAP)
TABLE_W = W - MARGIN - TABLE_X
REASON_W = TABLE_W - 96 - 56 - 60          # what is left for the reason column after detector, trip and padding
pages = []

p = Page("dashboard", "EV Battery Analytics")
p.text(0, 0, W, HEADER_H, [
    [("EV Battery Analytics", 17, True, "#FFFFFF"),
     ("   State-of-Charge Estimation · Energy Consumption Prediction · Anomaly Detection", 11, False, "#9FB3D1")],
    [("BMW i3 · 70 real trips around Munich · summer 2019 vs winter 2019/20 · 1.09 M samples at 10 Hz", 9, False, "#C8D3E6")]],
    background=NAVY)
p.text(MARGIN, H - FOOTER_H - 2, CONTENT_W, FOOTER_H,
       [[("Data: Battery and Heating Data in Real Driving Cycles (TU Munich, IEEE DataPort, CC BY 4.0) · models tested on "
          "14 unseen trips · a flag means worth an engineer's look, not a fault · built with Python, DuckDB and Power BI",
          8, False, MUTED)]])
p.season_slicer(MARGIN, ROW1_Y, SLICER_W, ROW1_H)
p.cards(MARGIN + SLICER_W + GAP, ROW1_Y, ROW1_H, [
    ("Trips", "Trips", INK), ("Consumption", "kWh per 100 km", NAVY),
    ("Heating Share", "Energy spent on heating", HEAT), ("Regen Share", "Recovered by braking", TEAL),
    ("SoC Model Error", f"SoC error, pts · R² {chosen.r2['A · SoC estimation']:.2f}", TEAL),
    ("Anomaly Flags", "Anomaly flags", FLAG)])

# ---------- row 2: energy and climate ----------
x = MARGIN
p.add("scatterChart", x, ROW2_Y, Q_W, ROW_H,
      roles={"Category": [C("dim_trip.trip_label", "Trip")], "Series": [C("dim_trip.season", "Season")],
             "X": [M("Ambient Temp", "Outside temperature (°C)")], "Y": [M("Consumption", "kWh per 100 km")]},
      title="Colder trips use more energy", subtitle="One dot per trip · kWh/100 km vs outside °C",
      objects={"dataPoint": SEASON_COLORS, "legend": [{"properties": {"show": lit("true"), "position": text_lit("Top")}}],
               "markers": [{"properties": {"markerSize": lit("6D")}}],
               "categoryAxis": [{"properties": {"start": lit("-5D"), "end": lit("35D"), "showAxisTitle": lit("true")}}],
               "valueAxis": [{"properties": {"showAxisTitle": lit("false")}}]})
x += Q_W + GAP
p.add("columnChart", x, ROW2_Y, Q_W, ROW_H,
      roles={"Category": [C("dim_trip.season", "Season")],
             "Y": [M("Traction", "Driving"), M("Heating", "Heating"), M("Air conditioning", "A/C")]},
      title="Where the energy goes",
      subtitle=f"kWh/100 km · top driver: {TOP.feature.lower()} ({TOP.share:.0%})",
      sort=(C("dim_trip.season"), "Ascending"),
      objects={"dataPoint": [measure_color("Traction", NAVY), measure_color("Heating", HEAT),
                             measure_color("Air conditioning", TEAL)],
               "labels": [{"properties": {"show": lit("true"), "color": color("#FFFFFF"), "labelPrecision": lit("1L")}}],
               "legend": [{"properties": {"show": lit("true"), "position": text_lit("Top")}}], **NO_AXIS_TITLES})
x += Q_W + GAP
p.add("lineClusteredColumnComboChart", x, ROW2_Y, Q_W, ROW_H,
      roles={"Category": [C("fact_warmup.minute", "Minutes since start")], "Y": [M("Heater Power", "Heater kW")],
             "Y2": [M("Cabin Temp", "Cabin °C")]},
      title="Winter warm-up", subtitle="Heater power peaks in the first minutes",
      sort=(C("fact_warmup.minute"), "Ascending"),
      objects={"dataPoint": [{"properties": {"fill": color(HEAT)}}, measure_color("Cabin Temp", NAVY)],
               "lineStyles": [{"properties": {"strokeWidth": lit("3D")}}],
               "legend": [{"properties": {"show": lit("true"), "position": text_lit("Top")}}],
               "categoryAxis": [{"properties": {"showAxisTitle": lit("true")}}],
               "valueAxis": [{"properties": {"showAxisTitle": lit("false"), "secShowAxisTitle": lit("false")}}]})
x += Q_W + GAP
p.add("clusteredColumnChart", x, ROW2_Y, Q_W, ROW_H,
      roles={"Category": [C("fact_regen_limit.batt_temp_band", "Battery temperature")],
             "Y": [M("Regen Limit", "Regen current (A)")]},
      title="A cold battery limits regeneration", subtitle="Peak braking current (A) by battery °C",
      sort=(C("fact_regen_limit.batt_temp_band"), "Ascending"),
      objects={"dataPoint": [{"properties": {"fill": color(TEAL)}}],
               "labels": [{"properties": {"show": lit("true"), "color": color(INK2)}}], **NO_AXIS_TITLES})

# ---------- row 3: battery health, SoC model, anomalies ----------
p.add("scatterChart", MARGIN, ROW3_Y, B_W, ROW_H,
      roles={"Category": [C("fact_sag_event.event_id", "Event")], "Series": [C("fact_sag_event.status", "Status")],
             "X": [M("Battery Temp", "Battery temperature (°C)")], "Y": [M("Resistance", "Pack resistance (mΩ)")]},
      title="Pack resistance falls as the battery warms", subtitle="Each dot is a heavy-load voltage dip · red = flagged",
      objects={"dataPoint": [value_color("fact_sag_event.status", "Normal", NEUTRAL),
                             value_color("fact_sag_event.status", "Flagged", FLAG)],
               "legend": [{"properties": {"show": lit("true"), "position": text_lit("Top")}}],
               "markers": [{"properties": {"markerSize": lit("5D")}}],
               "categoryAxis": [{"properties": {"start": lit("-5D"), "end": lit("35D"), "showAxisTitle": lit("true")}}],
               "valueAxis": [{"properties": {"showAxisTitle": lit("false")}}]})
p.add("lineChart", MARGIN + B_W + GAP, ROW3_Y, B_W, ROW_H,
      roles={"Category": [C("fact_soc_test.minute", "Minutes")],
             "Y": [M("SoC Recorded", "Recorded"), M("SoC Estimated", "Model estimate")]},
      title=f"Virtual SoC sensor · unseen trip {show_trip.replace('Trip', '')}",
      subtitle=f"Voltage, current, temperature only · error {err[show_trip]:.1f} pts",
      sort=(C("fact_soc_test.minute"), "Ascending"), filters=[in_filter("fact_soc_test.trip_id", [show_trip])],
      objects={"dataPoint": [measure_color("SoC Recorded", NAVY), measure_color("SoC Estimated", TEAL)],
               "lineStyles": [{"properties": {"strokeWidth": lit("2D")}}],
               "legend": [{"properties": {"show": lit("true"), "position": text_lit("Top")}}],
               "categoryAxis": [{"properties": {"showAxisTitle": lit("true")}}],
               "valueAxis": [{"properties": {"showAxisTitle": lit("false")}}]})
p.add("tableEx", TABLE_X, ROW3_Y, TABLE_W, ROW_H,
      roles={"Values": [C("fact_anomaly.detector_short", "Detector"), C("fact_anomaly.trip_id", "Trip"),
                        C("fact_anomaly.reason", "Why it was flagged")]},
      title=f"Anomaly register · {n_flags} flags with root-cause notes", subtitle="Scroll for all flags",
      sort=(C("fact_anomaly.detector_short"), "Ascending"),
      objects={"values": [{"properties": {"fontSize": lit("8D"), "fontColor": color(INK), "wordWrap": lit("true")}}],
               "columnWidth": [{"properties": {"value": lit("96D")}, "selector": {"metadata": "fact_anomaly.detector_short"}},
                               {"properties": {"value": lit("56D")}, "selector": {"metadata": "fact_anomaly.trip_id"}},
                               {"properties": {"value": lit(f"{REASON_W:.0f}D")}, "selector": {"metadata": "fact_anomaly.reason"}}],
               "columnHeaders": [{"properties": {"fontSize": lit("8D"), "fontColor": color(INK2), "bold": lit("true"),
                                                 "backColor": color(CANVAS)}}],
               "grid": [{"properties": {"gridHorizontal": lit("true"), "gridHorizontalColor": color(HAIRLINE),
                                        "rowPadding": lit("2D")}}]})
pages.append(p)
print({pg.display_name: len(pg.visuals) for pg in pages})

SoC example trip: TripB18 (mean error 1.9 points, median of 14 test trips)


{'EV Battery Analytics': 16}


## 4 · Write the report and the .pbip

In [7]:
RP = OUT / f"{NAME}.Report"
RDEF = RP / "definition"
theme_name = "theme_batteryinsights.json"
write_json(RP / "definition.pbir", {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definitionProperties/2.0.0/schema.json",
    "version": "4.0", "datasetReference": {"byPath": {"path": f"../{NAME}.SemanticModel"}}})
write_json(RP / ".platform", {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
    "metadata": {"type": "Report", "displayName": NAME},
    "config": {"version": "2.0", "logicalId": tag("logical", "report")}})
write_json(RDEF / "version.json", {"$schema": f"{SCHEMA}/versionMetadata/1.0.0/schema.json", "version": "2.0.0"})
version_at_import = {"visual": "1.8.97", "report": "2.0.97", "page": "1.3.97"}
write_json(RDEF / "report.json", {
    "$schema": f"{SCHEMA}/report/3.0.0/schema.json",
    "themeCollection": {
        "baseTheme": {"name": "CY24SU10", "reportVersionAtImport": version_at_import, "type": "SharedResources"},
        "customTheme": {"name": theme_name, "reportVersionAtImport": version_at_import, "type": "RegisteredResources"}},
    "resourcePackages": [
        {"name": "SharedResources", "type": "SharedResources",
         "items": [{"name": "CY24SU10", "path": "BaseThemes/CY24SU10.json", "type": "BaseTheme"}]},
        {"name": "RegisteredResources", "type": "RegisteredResources",
         "items": [{"name": theme_name, "path": theme_name, "type": "CustomTheme"}]}],
    "settings": {"useStylableVisualContainerHeader": True, "defaultFilterActionIsDataFilter": True,
                 "defaultDrillFilterOtherVisuals": True, "allowChangeFilterTypes": True, "useEnhancedTooltips": True}})
write_json(RP / "StaticResources" / "RegisteredResources" / theme_name, THEME)
write_json(ROOT / "powerbi" / theme_name, THEME)
write_json(RDEF / "pages" / "pages.json", {"$schema": f"{SCHEMA}/pagesMetadata/1.0.0/schema.json",
                                           "pageOrder": [pg.key for pg in pages], "activePageName": pages[0].key})
for pg in pages:
    write_json(RDEF / "pages" / pg.key / "page.json", pg.json())
    for v in pg.visuals:
        write_json(RDEF / "pages" / pg.key / "visuals" / v["name"] / "visual.json", v)
write_json(OUT / f"{NAME}.pbip", {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/pbip/pbipProperties/1.0.0/schema.json",
    "version": "1.0", "artifacts": [{"report": {"path": f"{NAME}.Report"}}], "settings": {"enableAutoRecovery": True}})
write(OUT / ".gitignore", "**/.pbi/localSettings.json\n**/.pbi/cache.abf\n")
print(f"{sum(1 for f in OUT.rglob('*') if f.is_file())} files written")

40 files written


## 5 · Validate every report JSON file against Microsoft's schemas

In [8]:
import requests
from jsonschema import Draft7Validator, Draft202012Validator
from referencing import Registry, Resource

_cache = {}


def fetch(url):
    if url not in _cache:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        _cache[url] = r.json()
    return _cache[url]


registry = Registry(retrieve=lambda uri: Resource.from_contents(fetch(uri)))
problems, checked = {}, 0
for path in sorted(OUT.rglob("*.json")) + sorted(OUT.rglob("*.pbir")) + sorted(OUT.rglob("*.pbism")) + sorted(OUT.rglob("*.pbip")):
    doc = json.loads(path.read_text(encoding="utf-8"))
    url = doc.get("$schema")
    if not url:
        continue
    schema = fetch(url)
    cls = Draft202012Validator if "2020-12" in schema.get("$schema", "") else Draft7Validator
    errors = sorted(cls(schema, registry=registry).iter_errors(doc), key=lambda e: list(e.path))
    checked += 1
    if errors:
        problems[path.relative_to(OUT).as_posix()] = [f"{list(e.path)}: {e.message[:200]}" for e in errors[:5]]
print(f"validated {checked} files against their $schema")
for f, errs in problems.items():
    print("x", f)
    for e in errs:
        print("   ", e)
assert not problems, "schema validation failed"
print("all report files are schema-valid")

validated 23 files against their $schema
all report files are schema-valid


## 6 · Open it

Open `powerbi/BatteryInsights/BatteryInsights.pbip` in Power BI Desktop, then click **Home → Refresh** to load the CSVs.
If the project was moved, update the `DataFolder` parameter (Transform data → Edit parameters) to the new `powerbi/data/` folder (keep the trailing backslash).